# Team Member Contribution Table

| Member      | Contribution |
| ----------- | ----------- |
| Petter (Scrum Master)      | Learned what we had done so far in the ml part such that I can be more involved in our discussions concerning ml. |
| Kim      | See Tree-based Classification: Random Forest vs Boosting (WeakestLink)|
| Elliot      | See Tree-Based Regression|
| Agust       | In A5_stacking.ipynb I done a stacking-only comparison for A5. The notebook loads bootstrap metric artifacts, selects two stacking variants (stacking_in_boot and stacking_passthrough_in_boot), and compares them for both regression and classification. For each metric it computes paired t-test, Wilcoxon test, effect size, bootstrap confidence intervals, and win rate.Then summarizes which stacking variant performs better and whether the difference is statistically significant. The second stacking variant (stacking_passthrough_in_boot) turned out to be the stronger overall candidate in my evaluation, especially for regression quality and classification probability quality. I have also been learning how the ML parts of our system fit together to prepare me and others that have been in SW to contribute more in ML for upcoming sprints.|

# Tree-based Classification: Random Forest vs Boosting (WeakestLink)

## Goal and sprint context
This work addresses the sprint issue:

> **“Challenge the current champion regression/classification with a tree-based approach.”**

In this notebook, the focus is **classification**: predicting the **WeakestLink** label using **tree-based ensemble methods**, comparing them against the **current champion** model. The objective is to see whether a tree-based approach can outperform the existing champion and, if so, identify which tree method should be promoted.

---

## Data and problem setup

### Data sources
Two processed datasets were used:
- `20190108_scores_and_weak_links.csv` — contains indicator columns used to derive the WeakestLink label (argmax over indicators).
- `AimoScore_WeakLink_big_scores.csv` — contains engineered features used as model inputs (Angle deviation + NASM deviation features).

### Target construction (WeakestLink)
The WeakestLink label was created by:
1. Selecting indicator columns (excluding non-indicators such as `ID`, `Date`, `SCORE`).
2. Optionally dropping a set of weak link classes (`DROP_WEAK_LINKS`) to match earlier project assumptions.
3. Computing each row’s maximum indicator value and labeling the row by the column (indicator) that achieved that maximum (i.e., `idxmax`).
4. Handling ties:
   - If multiple indicators tie for the maximum value, the row is marked as `HasTie=True`.
   - When `DROP_TIES=True`, rows with ties are removed to avoid ambiguous labels.

### Feature construction
After deriving the target:
- The label dataframe (`ID`, `WeakestLink`) is merged into the “big features” dataframe on `ID`.
- Feature columns are selected as:
  - All columns ending in `_Angle_Deviation`
  - All columns ending in `_NASM_Deviation`
- Rows with missing feature values are removed.
- The final dataset consists of:
  - `X`: numeric feature matrix (Angle + NASM deviations)
  - `y`: `WeakestLink` label (multi-class)

### Train/test split
A stratified split was used:
- `train_test_split(..., stratify=y, test_size=0.2, random_state=42)`

This ensures class proportions are roughly preserved between train and test.

---

## Modeling strategy

### Why tree ensembles?
Tree ensembles are strong candidates for this task because:
- They can model non-linear relationships and feature interactions.
- They handle heterogeneous feature effects without requiring scaling.
- Ensemble methods often improve generalization over a single decision tree.

### Models evaluated
To ensure a meaningful tree-based comparison, at least **3 forest-family** and **3 boosting-family** methods were tested.

#### Forest / bagging family
1. **RandomForestClassifier**
   - Bagging of decision trees with random feature selection at each split.
2. **ExtraTreesClassifier**
   - Like Random Forest, but adds more randomness by selecting split thresholds randomly.
3. **BaggingClassifier (DecisionTreeClassifier base)**
   - Pure bagging of decision trees (RF-like baseline without RF’s built-in split randomization and class weighting).

#### Boosting family
4. **HistGradientBoostingClassifier**
   - Gradient boosting using histogram-based binning for fast split search; usually strong and efficient.
5. **GradientBoostingClassifier**
   - Classic gradient boosting (sequential trees correcting errors of prior ensemble).
6. **AdaBoostClassifier (DecisionTree base)**
   - Boosting that reweights samples to focus subsequent learners on harder examples.

### Preprocessing (consistent across models)
Each model was wrapped in a pipeline:
- `SimpleImputer(strategy="median")` → model

This ensures missing values do not break training and makes preprocessing consistent across candidates.

### Hyperparameter tuning and evaluation
For each model:
- **RandomizedSearchCV** was used with:
  - 5-fold **StratifiedKFold**
  - optimization metric: **balanced_accuracy**
  - random seed fixed for reproducibility
- Best estimator was evaluated on the **held-out test set** using:
  - `accuracy`
  - `balanced_accuracy`
  - `macro_f1`
  - `weighted_f1`
  - `log_loss` (when compatible)

### MLflow tracking
Each tuned model:
- logged parameters (best hyperparameters),
- logged metrics (test metrics),
- logged the final best estimator to MLflow,
using the same MLflow connection strategy used in prior notebooks.

---

## Metrics: what they mean and why multiple are reported

Because WeakestLink classes are imbalanced, no single metric tells the full story.

- **Accuracy**
  - Overall correctness; can be misleading when classes are imbalanced.
- **Balanced accuracy**
  - Average recall across classes; reduces majority-class dominance.
- **Macro F1**
  - Average F1 across classes (each class equal weight); emphasizes rare class performance.
- **Weighted F1**
  - F1 weighted by class frequency; emphasizes majority-class performance while still reflecting precision/recall tradeoffs.
- **Log loss**
  - Evaluates probability quality; only comparable if label sets and probability columns match.

In this work, **macro F1** and **weighted F1** were emphasized because they reflect two important perspectives:
- macro F1: fairness across classes
- weighted F1: overall performance on the real class distribution

---

## Results summary (model comparison)

### Candidate ranking
The candidate models produced different tradeoffs across metrics. In general:
- **HistGradientBoostingClassifier** achieved the strongest overall performance on **weighted F1** and **macro F1** among tested candidates.
- **RandomForestClassifier** was strong and competitive, especially in weighted F1.
- **ExtraTreesClassifier** sometimes performed well on balanced accuracy (class fairness) but did not consistently dominate on F1.
- **BaggingClassifier** (bagged trees) was stable but tended to achieve lower macro F1, indicating consistently lower performance on minority classes.
- **AdaBoost** underperformed in this configuration, likely due to sensitivity to noise and class imbalance.
- **GradientBoostingClassifier** performed reasonably but typically did not exceed HistGradientBoosting.

### Comparison to the existing champion
The notebook also loaded and evaluated the existing champion (`champion_model.joblib`) on the current test set.

Important note:
- The saved champion model may have been trained with a slightly different label configuration or class set in earlier iterations.
- When labels/probabilities are not aligned, **log loss comparisons can become unreliable**. Accuracy/F1 comparisons remain usable as long as labels are consistent on the test set.

Overall, several of the tested tree-based candidates improved on the champion in at least one key metric (macro/weighted F1), suggesting that tree-based approaches are viable challengers.

---

![alt text](img/kde_macro_f1.png)
![alt text](img/kde_weighted_f1.png)


## KDE bootstrap analysis (interpretation)

### Why bootstrap + KDE?
To interpret performance differences beyond single test-set numbers, bootstrap resampling was used:

1. Predict the test set once per model.
2. Re-sample the test set (with replacement) many times.
3. Recompute the metric for each bootstrap resample.
4. Plot the distribution using **KDE** (Kernel Density Estimation).

This gives “normal distribution-like” curves that show:
- typical performance (where the curve sits on the x-axis),
- variability/stability (how narrow or wide it is),
- how much two models’ distributions overlap (uncertainty).

### How to read the KDE plot
For a given metric:
- **Curve farther right** → higher metric values → better performance.
- **Taller/narrower curve** → less variance → more stable performance.
- **Wider/flatter curve** → more variance → less stable performance.

Crucially:
- **Peak height is not “better” by itself**.
- A model can have a tall peak simply because it is consistently producing similar scores — even if those scores are low.

### Interpreting the observed KDE behavior
Two consistent patterns were observed:

1. **HistGradientBoosting tends to sit farthest to the right** for both weighted F1 and macro F1.
   - This indicates it has the highest typical F1 performance in the bootstrap analysis.

2. **Some weaker models (e.g., bagged trees) can show a tall peak**
   - This usually means low variance around a low metric value (consistently mediocre).
   - In practice, this is not a reason to choose them as champions.

### Quantitative interpretation from bootstrap summary
In addition to the KDE plots, bootstrap summaries were computed:
- mean, median, standard deviation
- 95% interval (2.5% to 97.5%)
- probability that one model beats another: `P(model_A > model_B)`

Example interpretation for macro F1:
- HistGradientBoosting had the highest median macro F1.
- `P(HGB > champion)` was around ~0.65 in the reported run.
  - This indicates HGB outperforms the champion in macro F1 in about 65% of bootstrap resamples.
  - The distributions still overlap substantially, so the improvement is positive but not absolute.

---

## Conclusion: should HistGradientBoosting become the new champion?

Based on the tree-based evaluation and bootstrap interpretation:

- **HistGradientBoostingClassifier** is the strongest overall candidate:
  - highest typical weighted F1
  - highest typical macro F1
  - strong test-set metrics among candidates
  - KDE distributions shifted right relative to others

However, the bootstrap comparison (e.g., `P(HGB > champion) ≈ 0.65` for macro F1) suggests the improvement is **moderate** rather than overwhelming in that metric alone.

## Notes and limitations
- The dataset is imbalanced; some classes may have few examples, affecting CV stability and bootstrap variability.
- The existing saved champion may be trained under a slightly different label configuration; probability-based comparisons (log loss) require aligned class sets.
- Results are based on one train/test split; repeated splits or nested CV would provide even stronger evidence, but bootstrap partially addresses uncertainty on the fixed test set.

---

## Final takeaway
Tree-based methods (especially **HistGradientBoosting**) provide a strong and practical improvement over the prior champion for WeakestLink classification under F1-based evaluation, making them suitable challengers and a reasonable new champion candidate.

# Tree-Based Regression

**Goal:** Improve upon the linear regression champion (Ridge, R² ≈ 0.607) by exploring tree-based models.

**Evaluation:** Test R², 10-fold cross-validation R² (mean ± std), paired t-test vs Random Forest baseline

---
## Models evaluated

| Model | Description |
|---|---|
| `RandomForest_baseline` | 200 trees, default sklearn settings |
| `RandomForest_tuned` | 1 000 trees, `max_features=0.5`, `min_samples_split=5`, `max_depth=40` — from randomised search |
| `XGBoost_baseline` | 500 trees, `lr=0.05`, `max_depth=6` |
| `XGBoost_exp_low_lr` | 1 000 trees, `lr=0.01`, `max_depth=6`, light regularisation |

---
## Results

All tree-based models substantially outperformed the Ridge champion (R² = 0.607).

| Model | CV R² mean | CV R² std |
|---|---|---|
| RandomForest baseline | 0.673 | 0.049 |
| RandomForest tuned | 0.697 | 0.0555 |
| XGBoost baseline | 0.709 | 0.060 |
| XGBoost exp (low lr) | 0.7152 | 0.062 |

---
## Random Forest: baseline vs tuned

To improve the baselineis, a randomised search over six hyperparameters was run with 50 iterations using 10-fold CV.

The best configuration found was:

```
n_estimators      = 1000
max_features      = 0.5        
min_samples_leaf  = 1
min_samples_split = 5          
max_depth         = 40         
max_samples       = None
```

The tuned model achieved **CV R² = 0.697** a statistically significant improvement over the baseline (0.673). The largest driver was `max_features=0.5`: using more features per split reduced the bias of individual trees, while `min_samples_split=5` and `max_depth=40` provided light regularisation against overfitting.

Even after tuning, the random forest is still outperformed by XGBoost.

![alt text](img/RFtunedvsBaseline.png)

The tuned Random Forest improvement vs the baseline was statistically significant with p < 0.05

---
## XGBoost experiments

Three XGBoost configurations were evaluated alongside the baseline.

**XGBoost baseline** (`lr=0.05`, 500 trees, `max_depth=6`) severely overfit the training set (train R² ≈ 1.00) while still achieving test R² = 0.760 — indicating the features carry genuine predictive signal despite the overfitting.

**XGBoost exp3** (lower learning rate `lr=0.01`, 1 000 trees, `max_depth=6`) achieved the best result — CV R² = 0.715.

Across all three experiments, **CV R² scores were nearly identical** (0.709–0.715). Paired t-tests showed no statistically significant difference between any XGBoost variant. The configurations converge to similar cross-validated performance, indicating that the data — rather than the exact hyperparameter choice — is the limiting factor at this stage.

All XGBoost experiments were significantly better than the Random Forest baseline (p < 0.001 in all cases), but none of the XGBoost experiments were significantly improved over each other.

---
## Feature removal experiment

To test whether noisy low-importance features were hurting XGBoost performance, the 5 features with the lowest gain-based importance scores were identified and dropped inside the pipeline (using a `ColumnDropper` transformer, so the drop is applied correctly within each CV fold and cannot leak information).

The removed features were: `No_2_Angle_Deviation`, `No_15_NASM_Deviation`, `No_16_NASM_Deviation`, `No_4_NASM_Deviation`, `No_1_Angle_Deviation`.

Results for `XGBoost_exp3_xgb_drop5`: CV R² mean = 0.7145, no meaningful change compared to the full-feature experiment (CV R² = 0.7152). The paired t-test between the two was not significant.

This result is expected: gradient boosted trees are inherently robust to irrelevant features, since splits are only made on features that reduce the loss. Low-importance features contribute little or nothing to the model, so removing them neither helps nor hurts meaningfully.

---
## Conclusions

1. All tree-based models significantly beat the last champion (R² = 0.607). The best models reach CV R² ≈ 0.7–0.715, a substantial improvement in predictive accuracy.

2. XGBoost outperforms Random Forest across all configurations. Even the XGBoost baseline outperforms the tuned Random Forest on CV R². Sequential boosting is better suited to this dataset than independent bagging.

3. Tuning Random Forest matters. The randomised search improved CV R² from 0.673 to 0.697, a statistically significant gain, with `max_features=0.5` as the primary driver.

4. XGBoost has plateaued with these features. All three XGBoost experiments converged to CV R² ≈ 0.71–0.72 regardless of hyperparameter configuration. Further gains are unlikely without new features or additional data.

5. Feature removal made no significant difference. The model is already ignoring low-importance features internally; explicitly removing them changes nothing.

###  Champion

**`XGBoost_exp_low_lr`** (`lr=0.01`, 1 000 trees, `max_depth=6`) had the best result with CV R² (0.7152). The lower learning rate makes it more stable and less prone to overfitting.

![alt text](img/XGBoost_vs_old_champion.png)
